### Importing Necessary Libraries and .csv files 

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.linear_model import LinearRegression, Ridge, Lasso, BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import warnings
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error


train_inputs = pd.read_csv('../datasets/cleaned datasets/train_inputs.csv')
train_targets = pd.read_csv('../datasets/cleaned datasets/train_targets.csv')

val_inputs = pd.read_csv('../datasets/cleaned datasets/val_inputs.csv')
val_targets = pd.read_csv('../datasets/cleaned datasets/val_targets.csv')

test_inputs = pd.read_csv('../datasets/cleaned datasets/test_inputs.csv')

submission_df = pd.read_csv('../datasets/initial kaggle datasets/sample_submission.csv')

In [20]:
train_inputs

,episode_number,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,total_popularity,popularity_difference,ad_density,peak_time_to_publish,is_night_or_evening,...,Publication_Day_Thursday,Publication_Day_Tuesday,Publication_Day_Wednesday,Publication_Time_Afternoon,Publication_Time_Evening,Publication_Time_Morning,Publication_Time_Night,Episode_Sentiment_Negative,Episode_Sentiment_Neutral,Episode_Sentiment_Positive
0,0.808081,0.274905,0.756130,0.5184,0.333333,0.624317,0.577135,0.016553,0,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,0.414141,0.198330,0.916413,0.9468,0.666667,0.929758,0.426598,0.045888,0,0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.535354,0.198330,0.891895,0.9733,0.000000,0.930955,0.398351,0.000000,0,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.484848,0.155885,0.968997,0.2247,0.333333,0.580786,0.857819,0.029191,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.787879,0.198330,0.932118,0.5182,0.000000,0.714658,0.674021,0.000000,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
562495,0.141414,0.198330,0.716008,0.4553,0.333333,0.570841,0.590228,0.022944,0,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
562496,0.797980,0.130703,0.709422,0.9654,1.000000,0.833064,0.302412,0.104446,1,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
562497,0.252525,0.225587,0.695542,0.8381,0.333333,0.759646,0.365703,0.020172,0,1,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
562498,0.030303,0.128766,0.351165,0.4519,0.333333,0.381567,0.391498,0.035339,1,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [21]:
train_targets

,Listening_Time_minutes
0,22.20941
1,18.16550
2,87.53108
3,55.17513
4,78.68007
...,...
562495,57.71737
562496,47.73335
562497,9.86572
562498,20.63115


## Before We Move onto Real ML models, we will start by creating dumb base models to help us evaluate how our future models will perform against it.

In [89]:
def predict_mean(inputs):
    return np.full(len(inputs), train_targets['Listening_Time_minutes'].mean())

In [88]:
root_mean_squared_error(predict_mean(train_inputs), train_targets), root_mean_squared_error(predict_mean(val_inputs), val_targets)

(27.148166175258353, 27.108639771954543)

In [90]:
def predict_median(inputs):
    return np.full(len(inputs), train_targets['Listening_Time_minutes'].median())

In [91]:
root_mean_squared_error(predict_median(train_inputs), train_targets), root_mean_squared_error(predict_median(val_inputs), val_targets)

(27.230040623248307, 27.194052431128263)

### Lets create a helper function to help us evaluate different types of base models really quickly.

In [34]:
def train_and_evaluate(ModelClass, **params):
    model = ModelClass(**params).fit(train_inputs, train_targets)
    train_rmse = root_mean_squared_error(model.predict(train_inputs), train_targets)
    val_rmse = root_mean_squared_error(model.predict(val_inputs), val_targets)
    print(f'Training RMSE: {round(train_rmse, 3)}')
    print(f'Validation RMSE: {round(val_rmse, 3)}')

In [92]:
train_and_evaluate(LinearRegression)

Training RMSE: 13.348
Validation RMSE: 13.353


In [102]:
train_and_evaluate(Ridge, random_state=42)

Training RMSE: 13.348
Validation RMSE: 13.352


In [107]:
train_and_evaluate(Lasso, random_state=42, tol=0.1)

Training RMSE: 14.041
Validation RMSE: 14.026


In [ ]:
warnings.filterwarnings('ignore')

train_and_evaluate(BayesianRidge, max_iter=300)

Training RMSE: 13.348
Validation RMSE: 13.352


In [ ]:
train_and_evaluate(DecisionTreeRegressor, max_depth=10, random_state=42)

Training RMSE: 13.074
Validation RMSE: 13.199


In [ ]:
warnings.filterwarnings('ignore')

train_and_evaluate(RandomForestRegressor, n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)

Training RMSE: 13.002
Validation RMSE: 13.114


### So far, our Random Forest Regressor Performs the best without any hyperparameter tuning. This gives us an idea that tree based models can work well with our dataset, so we will go on to use an XGBRegressor (Extreme Gradiant Boosting).

* Also, we will create a function to automate the creation of the submisssion file. Making predictions on the test data and embedding the results to the submission file.

In [4]:
def auto_submit(preds, fname):
    submission_copy = submission_df.copy()
    submission_copy['Listening_Time_minutes'] = preds
    submission_copy.to_csv(f'../submissions_to_make/{fname}.csv', index=None)
    print('Submission file has been created for the given model.')

### The below function is a manual hyperparameter tuner for a random forest regressor. It only tunes max_depth and max_leaf_nodes, but it could be extended as needed.

In [84]:
def overfitting_df_plot(ModelClass, max_depth_list, max_leaf_nodes_list, **params):
    import itertools
    from tqdm import tqdm
    values = []
    for i, j in tqdm(itertools.product(max_depth_list, max_leaf_nodes_list), desc='Training Model'):
        model = ModelClass(max_depth = i , max_leaf_nodes = j ,random_state=42, **params).fit(train_inputs, train_targets)

        train_rmse = root_mean_squared_error(model.predict(train_inputs), train_targets)
        val_rmse = root_mean_squared_error(model.predict(val_inputs), val_targets)

        values.append((i,j,train_rmse, val_rmse))

    df = pd.DataFrame(values, columns=['max_depth', 'max_leaf_nodes', 'train_rmse', 'val_rmse'])

    return df

In [116]:
overfitting_df_plot(DecisionTreeRegressor, [5,6,7,8,9,10], [i for i in range(20, 41)])

Training Model: 126it [08:45,  4.17s/it]


,max_depth,max_leaf_nodes,train_rmse,val_rmse
0,5,20,13.460315,13.452248
1,5,21,13.447784,13.443006
2,5,22,13.435963,13.430374
3,5,23,13.426282,13.420880
4,5,24,13.417579,13.413204
...,...,...,...,...
121,10,36,13.329885,13.330619
122,10,37,13.324926,13.326714
123,10,38,13.320272,13.323128
124,10,39,13.315337,13.316689


### First, lets try a simple XGBRegressor with no parameter tuning.

In [ ]:
train_and_evaluate(XGBRegressor, n_estimators=100, max_depth=5)

Training RMSE: 12.956
Validation RMSE: 13.068


In [118]:
model = XGBRegressor(n_estimators=100, max_depth=5, random_state=42).fit(train_inputs, train_targets)

In [121]:
xgbregressor_preds = model.predict(test_inputs)

In [123]:
auto_submit(xgbregressor_preds, 'xgbregressor1')

Submission file has been created for the given model.


### In order to determine the best hyperparameters, we will utilize RandomizedSearchCV for XGBRegressor.

In [ ]:
warnings.filterwarnings('ignore')

xgb_model = XGBRegressor()

param_grid = [{
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 6, 10],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1],
    'gamma': [0, 0.1, 0.2],
    'lambda': [0, 1, 10],
    'alpha': [0, 1, 10],
    'tree_method': ['auto', 'hist', 'gpu_hist']
}]


randomized_grid_search = RandomizedSearchCV(xgb_model, param_distributions=param_grid, n_iter=300 ,cv=3, n_jobs=-1, scoring='neg_root_mean_squared_error', verbose=2, random_state=42)
randomized_grid_search.fit(train_inputs, train_targets)

Fitting 3 folds for each of 300 candidates, totalling 900 fits


C:\Users\celik\AppData\Roaming\Python\Python39\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
324 fits failed out of a total of 900.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\celik\AppData\Roaming\Python\Python39\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\celik\AppData\Roaming\Python\Python39\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
  File "C:\Users\celik\AppData\Roaming\Python\Python39\site-packages\xgboost\sklearn.py", line 1170, in fit
    self._Booster = train(
  Fi

RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                   param_distributions=[{'alpha': [0, 1, 10],
                                         'colsample_bytree': [0.7, 0.8, 1],
                                         'gamma': [0, 0.1, 0.2],
                                         'lambda': [0, 1, 10],
                                         'learning_rate': [0.01, 0.05, 0.1,
                                                           0.2],
                                         'max_depth': [3, 6, 10],
                                         'min_child_weight': [1, 3, 5],
                                         'n_estimators': [100, 200, 500],
                                         'subsample': [0.7, 0.8, 1],
                                         'tree_method': ['auto', 'hist',
                                                         'gpu_hist']}],
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [132]:
grid_search.best_params_

{'tree_method': 'auto',
 'subsample': 0.8,
 'n_estimators': 500,
 'min_child_weight': 5,
 'max_depth': 10,
 'learning_rate': 0.05,
 'lambda': 10,
 'gamma': 0.1,
 'colsample_bytree': 1,
 'alpha': 1}

### Lets try our model with the hyperparameters provided bt the randomized search.

In [142]:
model = XGBRegressor(
    tree_method = 'auto',
    subsample = 0.8,
    n_estimators = 1000,
    min_child_weight = 5,
    max_depth = 12,
    learning_rate = 0.05,
    reg_lambda = 10,
    gamma = 0.1,
    colsample_bytree = 1,
    reg_alpha = 1
)
model.fit(train_inputs,train_targets)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None, colsample_bytree=1,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=12,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [143]:
root_mean_squared_error(model.predict(train_inputs), train_targets)

8.737415313720703

In [144]:
root_mean_squared_error(model.predict(val_inputs), val_targets)

12.74933910369873

### Now that we have a general idea of the optimal parameters to use, we will go above and beyond and try to optimize the most important parameters for overfitting/underfitting. Which are: max_depth, and n_estimators. The below function helps us automate this task.

In [ ]:
def overfitting_df_plot_xgb_regressor(ModelClass, max_depth_list, n_estimators_list, **params):
    import itertools
    from tqdm import tqdm
    values = []
    for i, j in tqdm(itertools.product(max_depth_list, n_estimators_list), desc='Training Model'):
        model = ModelClass(tree_method = 'auto',
                            subsample = 0.8,
                            n_estimators = j,
                            min_child_weight = 5,
                            max_depth = i,
                            learning_rate = 0.05,
                            reg_lambda = 10,
                            gamma = 0.1,
                            colsample_bytree = 1,
                            reg_alpha = 1,
                            n_jobs=-1).fit(train_inputs, train_targets)

        train_rmse = root_mean_squared_error(model.predict(train_inputs), train_targets)
        val_rmse = root_mean_squared_error(model.predict(val_inputs), val_targets)

        values.append((i,j,train_rmse, val_rmse))

    df = pd.DataFrame(values, columns=['max_depth', 'n_estimators_list', 'train_rmse', 'val_rmse'])

    return df

In [149]:
max_depth_values = [10, 11 ,12, 13, 14, 15]
n_estimator_values = [500, 1000, 1200, 1400, 1600]

overfitting_df_plot_xgb_regressor(XGBRegressor, max_depth_values, n_estimator_values)

Training Model: 30it [1:17:29, 154.99s/it]


,max_depth,max_leaf_nodes,train_rmse,val_rmse
0,10,500,11.604261,12.855729
1,10,1000,10.615486,12.801174
2,10,1200,10.265050,12.794991
3,10,1400,9.936015,12.791782
4,10,1600,9.626842,12.792895
5,11,500,11.063549,12.819818
6,11,1000,9.734830,12.774405
7,11,1200,9.270726,12.769464
8,11,1400,8.830926,12.770079
9,11,1600,8.433458,12.777225


In [ ]:
# best XGBRegressor values
# model = ModelClass(tree_method = 'auto',
#                             subsample = 0.8,
#                             n_estimators = 500,
#                             min_child_weight = 5,
#                             max_depth = 15,
#                             learning_rate = 0.05,
#                             reg_lambda = 10,
#                             gamma = 0.1,
#                             colsample_bytree = 1,
#                             reg_alpha = 1,
#                             n_jobs=-1)

#### So far, we have trained various models. Linear Regression, Decision Trees, RandomForests, and Gradient Boosters. The best models are models that ensemble these various algorithms and  generilize, utilize their advantages, while trying to diminish the disadvanteges. We will create an ensembler model which will help use do this for a given number of different models.

In [5]:
import numpy as np

class ModelEnsembler:
    def __init__(self, models, weights=None):
        self.models = models
        if weights is None:
            self.weights = np.ones(len(models)) / len(models)
        else:
            self.weights = np.array(weights) / np.sum(weights)

    def fit(self, inputs, targets):
        for model in self.models:
            model.fit(inputs, targets)
        return self
    
    def predict(self, inputs):
        preds = []
        for model in self.models:
            pred = model.predict(inputs)
            pred = pred.ravel()
            preds.append(pred)
        preds = np.array(preds)
        
        weighted_preds = np.average(preds, axis=0, weights=self.weights)
        return weighted_preds                                                                                                       
        

### Ensembling all the methods we have tried so far.

In [ ]:
warnings.filterwarnings('ignore')

# linear regressor
linear_regression = LinearRegression().fit(train_inputs, train_targets)

# xgbregressor
xgbregressor = XGBRegressor(tree_method= 'auto',
                            subsample = 0.8,
                            n_estimators = 500,
                            min_child_weight = 5,
                            max_depth = 15,
                            learning_rate = 0.05,
                            reg_lambda = 10,
                            gamma = 0.1,
                            colsample_bytree = 1,
                            reg_alpha = 1,
                            n_jobs=-1).fit(train_inputs, train_targets)

# lasso regressor
lasso = Lasso(random_state=42, alpha=0.1).fit(train_inputs, train_targets)

# ridge regressor
ridge = Ridge(random_state=42, alpha=0.1).fit(train_inputs, train_targets)

# bayesian ridge
bayesian_ridge = BayesianRidge().fit(train_inputs, train_targets)

# decision tree regressor
decision_tree_regressor = DecisionTreeRegressor(max_depth=10, random_state=42).fit(train_inputs, train_targets)

# random forest regressor
random_forest_regressor = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42).fit(train_inputs, train_targets)

# weights
model_weights = [0.025, 0.7, 0.025, 0.025, 0.025, 0.1, 0.1]

## ensembling the models
ensembled_model = ModelEnsembler(models=[linear_regression, xgbregressor, lasso, 
                                         ridge, bayesian_ridge, decision_tree_regressor,
                                         random_forest_regressor], weights=model_weights)

In [30]:
root_mean_squared_error(ensembled_model.predict(val_inputs), val_targets)

12.73917331540042

### Leaving out the weak models, and trying on the best 3 performers.

In [ ]:
# linear regressor
linear_regression = LinearRegression().fit(train_inputs, train_targets)

# xgbregressor
xgbregressor = XGBRegressor(tree_method= 'auto',
                            subsample = 0.8,
                            n_estimators = 500,
                            min_child_weight = 5,
                            max_depth = 15,
                            learning_rate = 0.05,
                            reg_lambda = 10,
                            gamma = 0.1,
                            colsample_bytree = 1,
                            reg_alpha = 1,
                            n_jobs=-1).fit(train_inputs, train_targets)

# decision tree regressor
decision_tree_regressor = DecisionTreeRegressor(max_depth=12, random_state=42).fit(train_inputs, train_targets)

# ensembling the models
ensembled_model = ModelEnsembler(models=[linear_regression, xgbregressor, decision_tree_regressor], weights=[0.1, 0.7, 0.2])

In [34]:
root_mean_squared_error(ensembled_model.predict(val_inputs), val_targets)

12.742127148563076

### Lets only try linear regression and xgbregressor

In [ ]:
# linear regression
linear_regression = LinearRegression().fit(train_inputs, train_targets)

# xgbregressor
xgbregressor = XGBRegressor(tree_method= 'auto',
                            subsample = 0.8,
                            n_estimators = 500,
                            min_child_weight = 3,
                            max_depth = 15,
                            learning_rate = 0.05,
                            reg_lambda = 10,
                            gamma = 0.1,
                            colsample_bytree = 1,
                            reg_alpha = 1,
                            n_jobs=-1,
                            random_state=42).fit(train_inputs, train_targets)

# ensembled model
ensembled_model = ModelEnsembler(models=[xgbregressor, linear_regression], weights=[0.8, 0.2])

In [35]:
root_mean_squared_error(ensembled_model.predict(val_inputs), val_targets)

12.70045803695969

In [ ]:
auto_submit(ensembled_model.predict(test_inputs), 'ensembled_model_lr_xgb')

Submission file has been created for the given model.


### This is our best model. From now on, we will try to optimize this model and maybe do some interesting feature engineering to decrease our RMSE.

### We will run a RandomizedSearchCV Algorithm once again, this time knowing what value of parameters are most likely to go well together from our past experiment, we can try to get a better set of them.

In [ ]:
warnings.filterwarnings('ignore')

xgb_model = XGBRegressor(learning_rate=0.05, subsample=0.8, n_estimators=500, reg_lambda=10,
                        gamma=0.1, colsample_bytree=1, reg_alpha=1, n_jobs=-1, random_state=42,
                        tree_method='hist')

param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_child_weight' : [3, 5, 7, 9] 
}


randomized_grid_search = RandomizedSearchCV(estimator= xgb_model, param_distributions= param_grid, n_iter=100, scoring='neg_root_mean_squared_error', verbose=2, random_state=42)
randomized_grid_search.fit(train_inputs, train_targets)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END ....................max_depth=5, min_child_weight=3; total time=  12.8s
[CV] END ....................max_depth=5, min_child_weight=3; total time=  13.4s
[CV] END ....................max_depth=5, min_child_weight=3; total time=  12.9s
[CV] END ....................max_depth=5, min_child_weight=3; total time=  24.3s
[CV] END ....................max_depth=5, min_child_weight=3; total time=  14.0s
[CV] END ....................max_depth=5, min_child_weight=5; total time=  14.1s
[CV] END ....................max_depth=5, min_child_weight=5; total time=  14.1s
[CV] END ....................max_depth=5, min_child_weight=5; total time=  14.6s
[CV] END ....................max_depth=5, min_child_weight=5; total time=  13.8s
[CV] END ....................max_depth=5, min_child_weight=5; total time=  14.4s
[CV] END ....................max_depth=5, min_child_weight=7; total time=  14.1s
[CV] END ....................max_depth=5, min_ch

RandomizedSearchCV(estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=1, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=0.1, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=0.05, max_...
                                          max_delta_step=None, max_depth=None,
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None, n_estimators=500,
                                          n_jobs=-1, num_parallel_tree=None,
                                          random_state=42, ...),
                   n_iter=100,
                   param_distributions={'max_depth': [5, 10, 15, 20],
                                        'min_child_weight': [3, 5, 7, 9]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [49]:
randomized_grid_search.best_params_

{'min_child_weight': 3, 'max_depth': 15}

### Lets initialize our model once again, now with the new parameter foundings from the RandomizedSearchCV

In [ ]:
# linear regression
linear_regression = LinearRegression().fit(train_inputs, train_targets)

# xgbregressor
xgbregressor = XGBRegressor(tree_method= 'auto',
                            subsample = 0.8,
                            n_estimators = 500,
                            min_child_weight = 3,
                            max_depth = 15,
                            learning_rate = 0.05,
                            reg_lambda = 10,
                            gamma = 0.1,
                            colsample_bytree = 1,
                            reg_alpha = 1,
                            n_jobs=-1,
                            random_state=42).fit(train_inputs, train_targets)

# ensembled model
ensembled_model = ModelEnsembler(models=[linear_regression, xgbregressor], weights=[0.1, 0.9])

In [15]:
root_mean_squared_error(ensembled_model.predict(val_inputs), val_targets)

12.806710917710115

In [55]:
auto_submit(ensembled_model.predict(test_inputs), 'ensembled4')

Submission file has been created for the given model.


In [13]:
pd.DataFrame({
    'features': train_inputs.columns,
    'importances': xgbregressor.feature_importances_
})

,features,importances
0,episode_number,0.004622
1,Episode_Length_minutes,0.155276
2,Host_Popularity_percentage,0.007068
3,Guest_Popularity_percentage,0.007086
4,Number_of_Ads,0.039853
...,...,...
82,Publication_Time_Morning,0.006552
83,Publication_Time_Night,0.006721
84,Episode_Sentiment_Negative,0.006210
85,Episode_Sentiment_Neutral,0.006357


### We achieved a top 25 percent on the leaderboard of the Kaggle Competition with the built ensembled model.